# 餐饮评论多数据类型分析

| 模块 | 数据类型 | 方法 | 运营价值 |
|---|---|---|---|
| Part 3-① | 表格数据 | EDA / 贝叶斯排名 / 相关分析 | 店铺客观排名 |
| Part 3-② | 序列数据 | 时段分析 / 差评爆发 / RFM | 排班优化 / 流失预警 |
| Part 3-③ | 图数据   | 菜品共现 / KOL识别 | 搭配推荐 / 口碑推广 |
| Part 4-① | 时空数据 | 城市热力 / 选址评分 | 新店选址决策 |
| Part 4-② | 文本数据 | 词频 / 维度分析 / BERT/ABSA | 差评溯源 / NLP训练集 |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

from analysis.data_loader import ReviewDataLoader
from analysis.preprocessor import ReviewPreprocessor
from analysis.analyzer import TabularAnalyzer, SequentialAnalyzer, GraphAnalyzer, SpatiotemporalAnalyzer
from analysis.text_analysis import TextAnalyzer
from analysis.strategy import AlertSystem, ReportGenerator

print('模块导入完成')

---
## 数据加载与预处理

In [ ]:
df_raw = ReviewDataLoader.from_csv('../data/sample_reviews.csv')
ReviewDataLoader.describe(df_raw)

In [ ]:
df = ReviewPreprocessor().fit_transform(df_raw)
print(f'预处理后: {len(df)} 条  列数: {df.shape[1]}')
df[['shop_name','city','review_score','sentiment','order_type','content_len']].head(5)

---
## Part 3-① 表格数据 (Tabular Data)

In [ ]:
ta = TabularAnalyzer(df)
stats = ta.basic_stats()
print('基础统计:')
for k, v in stats.items():
    print(f'  {k}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 评分分布
sd = ta.score_distribution()
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
axes[0].bar(sd['星级'].astype(str), sd['评论数'], color=colors)
for i, row in sd.iterrows():
    axes[0].text(i, row['评论数']+3, f"{row['占比%']}%", ha='center', fontsize=9)
axes[0].set_title('评分分布', fontsize=13)
axes[0].set_xlabel('星级')
axes[0].set_ylabel('评论数')

# 情感分布
sent = ta.basic_stats()['情感分布']
label_cn = {'positive':'正面','negative':'负面','neutral':'中性'}
labels = [label_cn.get(k,k) for k in sent.keys()]
axes[1].pie(list(sent.values()), labels=labels,
            colors=['#27ae60','#e74c3c','#95a5a6'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('情感分布', fontsize=13)

# 消费类型
od = ta.order_type_stats()
if not od.empty:
    axes[2].barh(od['order_type'], od['评论数'], color='#3498db')
    axes[2].set_title('消费类型', fontsize=13)

plt.tight_layout()
plt.savefig('../outputs/figures/01_basic.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 店铺综合排名（贝叶斯平滑）
ranking = ta.shop_ranking()
print('店铺综合排名:')
display(ranking)

# 排名可视化
fig, ax = plt.subplots(figsize=(10, 5))
colors_rank = ['#f39c12' if g=='S' else '#27ae60' if g in ('A',) else '#3498db' if g=='B' else '#e74c3c'
               for g in ranking['评级'].astype(str)]
bars = ax.barh(ranking['shop_name'], ranking['综合分'], color=colors_rank)
ax.set_xlabel('综合评分')
ax.set_title('店铺综合排名（贝叶斯平滑 + 差评率 + 活跃度）', fontsize=12)
for bar, row in zip(bars, ranking.itertuples()):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'[{row.评级}] 差评{row.差评率}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/figures/02_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 相关性热力图
corr = ta.correlation_matrix()
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('特征相关性热力图', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../outputs/figures/03_corr.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3-② 序列数据 (Sequential Data)

In [ ]:
sa = SequentialAnalyzer(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 时段分析
hourly = sa.hourly_stats()
ax1 = axes[0]
ax1.bar(hourly['小时'], hourly['评论数'], color='#3498db', alpha=0.8)
ax2 = ax1.twinx()
ax2.plot(hourly['小时'], hourly['差评率%'], 'r--o', markersize=4, label='差评率%')
ax1.set_title('各时段评论量 vs 差评率', fontsize=12)
ax1.set_xlabel('小时')
ax1.set_ylabel('评论数', color='#3498db')
ax2.set_ylabel('差评率 (%)', color='red')
ax2.legend(loc='upper right')

# 星期分析
weekly = sa.weekday_stats()
bar_colors = ['#e74c3c' if d in ['周六','周日'] else '#3498db' for d in weekly['星期']]
axes[1].bar(weekly['星期'], weekly['评论数'], color=bar_colors)
axes[1].set_title('星期评论分布（红=周末）', fontsize=12)

plt.tight_layout()
plt.savefig('../outputs/figures/04_time.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n营业时间优化建议:')
display(hourly[['小时','评论数','差评率%','建议']].head(24))

In [ ]:
# 差评爆发检测
burst = sa.detect_burst()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(burst['_date'], burst['差评率'], color='#3498db', linewidth=1.5, label='每日差评率')
crisis = burst[burst['风险'].isin(['危机','预警'])]
ax.scatter(crisis['_date'], crisis['差评率'], color='red', zorder=5, s=60, label='异常点')
ax.axhline(y=0.35, color='orange', linestyle='--', alpha=0.7, label='预警线 35%')
ax.set_title('差评率时序（Z-Score 异常检测）', fontsize=12)
ax.set_xlabel('日期')
ax.set_ylabel('差评率')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/05_burst.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'异常时间点: {len(crisis)} 个')

In [ ]:
# RFM 用户分层
rfm = sa.rfm_segments()
seg_ct = rfm['用户类型'].value_counts()
print('用户分层统计:')
for seg, cnt in seg_ct.items():
    print(f'  {seg}: {cnt} 人 ({cnt/len(rfm)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RFM 散点图
seg_color = {'高价值活跃':'#f1c40f','普通活跃':'#2ecc71',
             '沉睡用户':'#95a5a6','流失风险':'#e67e22','已流失':'#e74c3c'}
for seg, grp in rfm.groupby('用户类型'):
    axes[0].scatter(grp['距今天数'], grp['评论次数'],
                    c=seg_color.get(seg,'#3498db'), label=seg, alpha=0.7, s=60)
axes[0].set_xlabel('距今天数 (Recency)')
axes[0].set_ylabel('评论频次 (Frequency)')
axes[0].set_title('RFM 用户分层散点图', fontsize=12)
axes[0].legend(fontsize=8)

# 用户类型分布饼图
axes[1].pie(seg_ct.values, labels=seg_ct.index,
            colors=[seg_color.get(k,'gray') for k in seg_ct.index],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('用户类型分布', fontsize=12)

plt.tight_layout()
plt.savefig('../outputs/figures/06_rfm.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n挽留建议 (前10):')
display(rfm[['用户名','用户类型','距今天数','评论次数','平均分','运营建议']].head(10))

---
## Part 3-③ 图数据 (Graph Data)

In [ ]:
ga = GraphAnalyzer(df)

# 菜品口碑
dish_sent = ga.dish_sentiment()
print('菜品口碑分析:')
display(dish_sent.head(12))

if not dish_sent.empty and len(dish_sent) >= 3:
    top = dish_sent.head(12)
    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(top))
    w = 0.35
    ax.bar([i-w/2 for i in x], top['正评率%'], w, label='正评率%', color='#27ae60', alpha=0.85)
    ax.bar([i+w/2 for i in x], top['负评率%'], w, label='负评率%', color='#e74c3c', alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(top['菜品'], rotation=30, ha='right')
    ax.set_title('热门菜品正负评率对比', fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig('../outputs/figures/07_dish.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 菜品共现（推荐搭配）
cooccur = ga.dish_cooccurrence(top_n=15)
print('菜品共现组合（推荐搭配）:')
display(cooccur.head(12))

# KOL 用户
kol = ga.influencer_users(top_n=10)
print('\n高影响力用户 (KOL):')
display(kol)

---
## Part 4-① 时空数据 (Spatiotemporal Data)

In [ ]:
geo = SpatiotemporalAnalyzer(df)
city_sum = geo.city_summary()
site = geo.site_score()

print('城市维度分析:')
display(city_sum)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(city_sum['city'], city_sum['评论数'], color='#3498db')
axes[0].set_title('各城市评论量', fontsize=12)
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(city_sum['city'], city_sum['平均分'], color='#f39c12')
axes[1].set_ylim(0, 5)
axes[1].set_title('各城市平均评分', fontsize=12)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../outputs/figures/08_geo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 时空热力矩阵
heatmap = geo.spatiotemporal_heatmap()
if not heatmap.empty:
    plt.figure(figsize=(11, 4))
    sns.heatmap(heatmap.T, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
    plt.title('城市 × 时段 评论热力矩阵', fontsize=12)
    plt.tight_layout()
    plt.savefig('../outputs/figures/09_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

print('\n选址综合评分:')
display(site)

---
## Part 4-② 非结构化文本数据 (Text Data)

In [ ]:
txt = TextAnalyzer()

# 五维度覆盖率
dim_cov = txt.dimension_coverage(df)
print('用户评论维度覆盖率:')
display(dim_cov)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(dim_cov['维度'], dim_cov['提及率%'], color='#9b59b6')
axes[0].set_title('维度提及率 (%)', fontsize=12)
axes[1].barh(dim_cov['维度'], dim_cov['维度差评%'],
             color=['#e74c3c' if v > 30 else '#f39c12' for v in dim_cov['维度差评%']])
axes[1].set_title('维度差评率 (%)', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/10_dim.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 正/负词频对比
pos_w = txt.word_frequency(df, sentiment='positive', top_n=15)
neg_w = txt.word_frequency(df, sentiment='negative', top_n=15)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(pos_w['词语'][::-1], pos_w['频次'][::-1], color='#27ae60')
axes[0].set_title('正面评论高频词', fontsize=12)
axes[1].barh(neg_w['词语'][::-1], neg_w['频次'][::-1], color='#e74c3c')
axes[1].set_title('负面评论高频词', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/11_words.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n差评高频信号词:')
display(txt.neg_signal_words(df, top_n=10))

In [ ]:
# BERT 数据集
bert_ds = txt.build_bert_dataset(df)
print('BERT 训练集样例:')
display(bert_ds['train'].head(5))

# ABSA 数据集
absa_ds = txt.build_absa_dataset(df)
print('\nABSA 标注样例:')
display(absa_ds.head(8))

# 保存
txt.save_bert_dataset(bert_ds, output_dir='../outputs/datasets')
if not absa_ds.empty:
    txt.save_absa_pyabsa(absa_ds, '../outputs/datasets/absa_train.txt')

---
## 运营策略：差评预警 + 综合报告

In [ ]:
alert = AlertSystem(df)
alert_df = alert.generate()
print('差评预警列表:')
display(alert_df[['预警等级','shop_name','平均分','评分趋势','7日差评%','7日评论数']].head(12))

In [ ]:
# 预警可视化
fig, ax = plt.subplots(figsize=(10, 5))
level_color = {'P0 危机':'#e74c3c','P1 预警':'#e67e22','P2 关注':'#f1c40f','P3 跟踪':'#2ecc71','正常':'#95a5a6'}
colors_al = [level_color.get(r['预警等级'],'#95a5a6') for _, r in alert_df.iterrows()]
ax.barh(alert_df['shop_name'], alert_df['7日差评%'], color=colors_al)
ax.axvline(x=25, color='orange', linestyle='--', label='P2关注线 25%')
ax.axvline(x=35, color='red',    linestyle='--', label='P1预警线 35%')
ax.set_xlabel('7日差评率 (%)')
ax.set_title('各店铺差评率预警', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/12_alert.png', dpi=150, bbox_inches='tight')
plt.show()

# 打印预警报告文本
print(alert.text_report())

In [ ]:
# 生成综合运营报告
gen = ReportGenerator(output_dir='../outputs')
report_data = {
    'shop_ranking':   ranking.to_dict(orient='records'),
    'dish_sentiment': dish_sent.head(15).to_dict(orient='records'),
    'city_summary':   city_sum.to_dict(orient='records'),
    'dim_coverage':   dim_cov.to_dict(orient='records'),
    'alert_text':     alert.text_report(),
}
report = gen.build(df, report_data)

gen.to_json(report)
gen.to_excel(report)
gen.to_markdown(report)

print('\n已生成文件:')
for f in os.listdir('../outputs'):
    if not os.path.isdir(os.path.join('../outputs', f)):
        print(f'  {f}')